In [1]:
import pandas as pd

In [2]:
wdi = pd.read_csv("data/wdi_panel.csv")
imf = pd.read_csv("data/imf_panel.csv")

In [3]:
wdi_countries = set(wdi["country_code"].unique())
imf_countries = set(imf["country_code"].unique())

print(f"WDI: {len(wdi_countries)} países")
print(f"FMI: {len(imf_countries)} países")

WDI: 117 países
FMI: 211 países


In [4]:
print("WDI:", wdi["year"].min(), "-", wdi["year"].max())
print("FMI:", imf["year"].min(), "-", imf["year"].max())

WDI: 2010 - 2025
FMI: 2010 - 2026


In [5]:
wdi_countries = set(wdi["country_code"].unique())
imf_countries = set(imf["country_code"].unique())

print(f"WDI: {len(wdi_countries)}")
print(f"FMI: {len(imf_countries)}")

WDI: 117
FMI: 211


In [6]:
test_merge = wdi.merge(
    imf,
    on=["country_code", "year"],
    how="left",
    indicator=True
)

test_merge["_merge"].value_counts()

_merge
both          1872
left_only        0
right_only       0
Name: count, dtype: int64

In [7]:
print("WDI:", wdi.shape)
print("Teste:", test_merge.shape)

WDI: (1872, 17)
Teste: (1872, 20)


In [8]:
panel_final = wdi.merge(
    imf,
    on=["country_code", "year"],
    how="left"
)

In [10]:
missing_pct = (
    panel_final
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print(missing_pct)

country_code                       0.0
country                            0.0
year                               0.0
gdp_per_capita                     0.0
real_gdp_growth                    0.0
inflation_rate                     0.0
compulsory_education_years         0.0
population_65_plus_pct             0.0
unemployment_rate                  0.0
government_expenditure_pct_gdp     0.0
government_revenue_pct_gdp         0.0
gdp_deflator_inflation             0.0
age_dependency_ratio               0.0
gross_capital_formation_pct_gdp    0.0
total_reserves_months_imports      0.0
savings_pct_gdp                    0.0
exports_pct_gdp                    0.0
fiscal_balance_pct_gdp             0.0
gross_debt_pct_gdp                 0.0
dtype: float64


In [11]:
problem_countries = (
    panel_final.loc[
        panel_final.drop(
            columns=["country", "country_code", "year"],
            errors="ignore"
        ).isna().any(axis=1),
        "country_code"
    ]
    .unique()
    .tolist()
)

print(problem_countries)

[]


In [12]:
panel_final = panel_final[
    ~panel_final["country_code"].isin(problem_countries)
].copy()

In [13]:
desc = panel_final.describe().T

print(desc)

                                  count          mean           std  \
year                             1872.0   2017.500000      4.611004   
gdp_per_capita                   1872.0  18715.133675  23292.921081   
real_gdp_growth                  1872.0      3.105191      4.895782   
inflation_rate                   1872.0      6.629896     20.981916   
compulsory_education_years       1872.0     10.069444      2.372677   
population_65_plus_pct           1872.0     10.275435      6.501084   
unemployment_rate                1872.0      8.049698      5.779575   
government_expenditure_pct_gdp   1872.0     26.707003     10.622857   
government_revenue_pct_gdp       1872.0     25.569577      9.800364   
gdp_deflator_inflation           1872.0      8.316636     38.454580   
age_dependency_ratio             1872.0     56.435028     14.638028   
gross_capital_formation_pct_gdp  1872.0     23.954813      6.780378   
total_reserves_months_imports    1872.0      4.862739      4.253273   
saving

In [14]:
panel_final.to_csv(
    "data/df_model_panel.csv",
    index=False
)